<a href="https://colab.research.google.com/github/djqx7/iasnlp_2026/blob/main/llm/text_only_input_ipynpb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" sentencepiece scikit-learn pandas tqdm huggingface_hub

In [2]:
import torch
import bitsandbytes as bnb

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

print("bitsandbytes version:", bnb.__version__)
from google.colab import files

uploaded = files.upload()

GPU available: True
GPU name: Tesla T4
bitsandbytes version: 0.49.2


Saving test_dataset_final.csv to test_dataset_final (2).csv


In [5]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from google.colab import files

file_path = "test_dataset_final.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.columns)

text_col = "transcript"
label_col = "label"

df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

model_name = "Qwen/Qwen2.5-3B-Instruct"

use_gpu = torch.cuda.is_available()

if not use_gpu:
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

model.eval()

def make_prompt(sent):
    prompt = f"""You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Sentence: {sent}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Label:"""

    return prompt

def predict(sent):
    prompt = make_prompt(sent)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    ans = tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    ans_low = ans.lower()

    for lab in labels:
        if lab.lower() == ans_low:
            return lab

    for lab in labels:
        if lab.lower() in ans_low:
            return lab

    return ans

preds = []

for sent in tqdm(df[text_col].astype(str).tolist()):
    preds.append(predict(sent))

df["predicted_type"] = preds

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("qwen_predictions.csv", index=False)

print("Saved predictions to qwen_predictions.csv")

files.download("qwen_predictions.csv")

      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Index(['file_name', 'transcript', 'label'], dtype='str')
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
  0%|          | 0/404 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
100%|██████████| 404/404 [05:34<00:00,  1.21it/s]

Accuracy: 0.6732673267326733
Precision: 0.6994706710053245
Recall: 0.6732673267326733
F1 Score: 0.6709276669910452

Classification Report:

               precision    recall  f1-score   support

  declarative       0.55      0.39      0.46       106
  exclamatory       0.91      0.69      0.78       102
   imperative       0.51      0.80      0.62        99
interrogative       0.83      0.85      0.84        97

     accuracy                           0.67       404
    macro avg       0.70      0.68      0.67       404
 weighted avg       0.70      0.67      0.67       404

Saved predictions to qwen_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
from huggingface_hub import login
from huggingface_hub import login

login()
login()

In [8]:
import pandas as pd
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
from google.colab import files

file_path = "test_dataset_final.csv"

df = pd.read_csv(file_path)

print(df.head())
print(df.columns)

text_col = "transcript"
label_col = "label"

df = df.dropna(subset=[text_col, label_col]).reset_index(drop=True)

labels = sorted(df[label_col].astype(str).unique().tolist())

print("Text column:", text_col)
print("Label column:", label_col)
print("Labels:", labels)

model_name = "google/gemma-2-2b-it"

use_gpu = torch.cuda.is_available()

if not use_gpu:
    raise RuntimeError("Please enable GPU in Colab: Runtime -> Change runtime type -> GPU")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

model.eval()

def make_prompt(sent):
    prompt = f"""You are a sentence type classifier.

Classify the sentence into exactly one of these labels:

declarative: a normal statement or information-giving sentence.
interrogative: a question or sentence asking for information.
imperative: a command, request, advice, instruction, or order.
exclamatory: a sentence showing strong emotion, surprise, excitement, or emphasis.

Examples:
Sentence: I am going to school today.
Label: declarative

Sentence: What are you doing here?
Label: interrogative

Sentence: Please close the door.
Label: imperative

Sentence: What a beautiful day!
Label: exclamatory

Sentence: Give me scotch, please.
Label: imperative

Sentence: I heard you and James are engaged at last.
Label: declarative

Sentence: Hi, what brings you to my office today?
Label: interrogative

Now classify this sentence.

Sentence: {sent}

Return only one label from:
declarative, exclamatory, imperative, interrogative

Label:"""

    return prompt

def predict(sent):
    prompt = make_prompt(sent)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=10,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    ans = tokenizer.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    ans_low = ans.lower()

    for lab in labels:
        if lab.lower() == ans_low:
            return lab

    for lab in labels:
        if lab.lower() in ans_low:
            return lab

    return ans

preds = []

for sent in tqdm(df[text_col].astype(str).tolist()):
    preds.append(predict(sent))

df["predicted_type"] = preds

y_true = df[label_col].astype(str).tolist()
y_pred = df["predicted_type"].astype(str).tolist()

acc = accuracy_score(y_true, y_pred)

p, r, f1, _ = precision_recall_fscore_support(
    y_true,
    y_pred,
    average="weighted",
    zero_division=0
)

print("Accuracy:", acc)
print("Precision:", p)
print("Recall:", r)
print("F1 Score:", f1)

print("\nClassification Report:\n")
print(classification_report(y_true, y_pred, zero_division=0))

df.to_csv("gemma_predictions.csv", index=False)

print("Saved predictions to gemma_predictions.csv")

files.download("gemma_predictions.csv")

      file_name                                         transcript  \
0   0_0_d45.wav     Hello, Jane. I'm really glad to meet you here.   
1   0_0_d47.wav         I heard you and James are engaged at last.   
2  0_0_d215.wav                            Give me scotch, please.   
3   0_0_d49.wav            Hi, what brings you to my office today?   
4  0_0_d159.wav  Please help yourself at your dishes. I hope yo...   

           label  
0    declarative  
1    declarative  
2     imperative  
3  interrogative  
4     imperative  
Index(['file_name', 'transcript', 'label'], dtype='str')
Text column: transcript
Label column: label
Labels: ['declarative', 'exclamatory', 'imperative', 'interrogative']


config.json:   0%|          | 0.00/838 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]


  0%|          | 0/404 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)

100%|██████████| 404/404 [05:19<00:00,  1.27it/s]

Accuracy: 0.7128712871287128
Precision: 0.7173728794206372
Recall: 0.7128712871287128
F1 Score: 0.7067411155240725

Classification Report:

               precision    recall  f1-score   support

  declarative       0.59      0.58      0.58       106
  exclamatory       0.93      0.79      0.86       102
   imperative       0.65      0.52      0.57        99
interrogative       0.71      0.98      0.82        97

     accuracy                           0.71       404
    macro avg       0.72      0.72      0.71       404
 weighted avg       0.72      0.71      0.71       404

Saved predictions to gemma_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>